In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [2]:
# BiLSTM training for 12-hour wind vector forecasting

# This notebook trains `BiLSTMModel` on 10-min measurements to forecast the next 12 hours (72 steps)
# of u/v vector components per station.

import os
from omegaconf import OmegaConf
from loguru import logger
import pandas as pd

from src.database.database_service import DatabaseService
from src.weather_stations.weather_station_service import WeatherStationService
from src.measurements.measurement_service import MeasurementService
from src.model.variant.bilstm_model import BiLSTMModel


/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load config and initialize services
cfg = OmegaConf.load("conf/config.yaml")

db = DatabaseService(cfg)
ws_service = WeatherStationService(cfg, db)
weather_stations = ws_service.load_from_database(only_relevant=True)

ms_service = MeasurementService(cfg, db, weather_stations)

logger.info(f"Loaded {len(weather_stations)} weather stations")


2025-09-10 12:10:41.765 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:228 - Loaded 50 weather stations from database
2025-09-10 12:10:41.788 | INFO     | __main__:<module>:10 - Loaded 50 weather stations


In [ ]:
# Load measurements from DB
measurements_df = ms_service.load_all_measurements_from_database()
logger.info(f"Measurements: {len(measurements_df)} rows")

# Optional: downsample or filter timeframe for quicker training
start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2024-12-31')
train_df = measurements_df[(measurements_df['record_date']>=start_date) & (measurements_df['record_date']<=end_date)].copy()

test_df = measurements_df[measurements_df['record_date']>end_date].copy()

train_df.head()


2025-09-10 12:12:48.950 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 1000000)
2025-09-10 12:13:01.119 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 2000000)
2025-09-10 12:13:12.972 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 3000000)


In [ ]:
# Instantiate BiLSTM model
model = BiLSTMModel(
    history_steps=72,   # 12 hours of history
    horizon_steps=72,   # 12 hours forecast
    station_embedding_dim=8,
    hidden_size=128,
    num_layers=2,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    num_epochs=5,   # increase for real training
)
model


In [ ]:
# Train model
model.train(train_df)


In [ ]:
# Save model
save_dir = "models/bilstm"
os.makedirs(save_dir, exist_ok=True)
model.save(save_dir)
logger.info(f"Saved BiLSTM model to {save_dir}")


In [ ]:
# Quick prediction demo for a subset (uses last 72 steps per station)
preds = model.predict(measurements_df)
preds.head()
